### imports specifically for kaggle

In [1]:

!pip install openai==1.30.1 
!pip install langchain==0.2.14 langchain-openai==0.1.22
!pip install numpy==2.0.0 pandas==2.2.2 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 6.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.82.4 requires openai>=2.8.0, but you have openai 1.30.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.8/997.8 kB 17.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.4 MB/s eta 0:00:00:00:

In [ ]:
import json
import re
from pathlib import Path

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import JsonOutputParser

from pydantic import BaseModel, Field
from typing import List, Optional
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login




In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN2")

login(token=hf_token)

In [4]:
llm = ChatOpenAI(
    model="meta-llama/Llama-3.1-8B-Instruct",
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token
)



In [5]:
INPUT_FILE = Path("/kaggle/input/datasets/marwanelnabawy/big-data/cleaned_products.json")
OUTPUT_FILE = Path("/kaggle/working/enriched_products.json")

SYSTEM_PROMPT = """You are a senior data engineer and product intelligence system.\
You will receive a LIST of products product(JSON array) with messy or partially incorrect fields.\
Your task is to clean, correct, and enrich the product into a fully structured format.\
"""

USER_PROMPT_TEMPLATE = """
For EACH product:
- Clean and normalize fields
- Fix brand and category hierarchy
- Don't add any fields exept the ones given
RULES:
- Return ONLY valid JSON (no text, no explanation,no functions and no javascript) and no "```json"
- Never return AImessage or content or meta data, just the list of JSON objects
- Keep same order as input, and don't dublicate outputs
- Prefer information from name first, then description
FORMAT INSTRUCTIONS:{format_instructions}
INPUT:{input_json}
"""

In [6]:
with INPUT_FILE.open("r", encoding="utf-8") as f:
    products = json.load(f)

In [7]:
print("Total products:", len(products))

for p in products[:2]:
    print(json.dumps(p, indent=2, ensure_ascii=False))


Total products: 286
{
  "product_id": "25804",
  "name": "Juhayna Full Cream Milk - 1Kg",
  "brand": "Juhayna",
  "category": "Full",
  "price": 44.99,
  "original_price": 52.75,
  "discount": 14.7,
  "availability": "in_stock",
  "description": "Juhayna Full Cream Milk - 1Kg"
}
{
  "product_id": "11760",
  "name": "Crystal Glass Cleaner Spray700+Refil S.O",
  "brand": "Crystal",
  "category": "Glass",
  "price": 66.95,
  "original_price": 79.95,
  "discount": 16.3,
  "availability": "in_stock",
  "description": "Crystal Glass Cleaner Spray700+Refil S.O"
}


In [8]:
class Size(BaseModel):
    value: Optional[float] = Field(None, description="Numeric value of the size (e.g., 500)")
    unit: Optional[str] = Field(None, description="Unit of measurement: g, kg, ml, L, pcs")

class Price(BaseModel):
    original_price: float = Field(..., description="Original price before discount")
    discount_percentage: Optional[float] = Field(0, description="Discount percentage (0 if no discount)")
    current_price: Optional[float] = Field( None, description="Final price after applying discount.If discount_percentage is 0, equals original.")

class Product(BaseModel):
    product_id: str = Field(..., description="Unique product identifier")

    name: str = Field(..., description="Cleaned product name")
    brand: str = Field(..., description="Product brand extacted from name or description")

    category: str = Field(..., description="Top-level category")
    product_type: str = Field(..., description="Specific product type")

    size: Optional[Size] = Field(None, description="Product size info")
    price: Price = Field(..., description="Pricing information")

    availability: str = Field(..., description="Stock status: in_stock or out_of_stock")

    description: str = Field(..., description="Short product description")
    tags: List[str] = Field(..., description="List of relevant tags extacted from name or description")

In [9]:
parser = JsonOutputParser(pydantic_object=Product)
format_instructions = parser.get_format_instructions()

In [10]:
results = []

def is_valid_item(item, parser) :
    if not isinstance(item, dict):
        return False
    required = {"product_id", "name", "brand", "price","availability","description","tags","category","product_type"}
    return all(key in item for key in required)

def run_pipeline(chain, products, batch_size=20, max_retries=3):
    for i in range(0, len(products), batch_size):
        batch = products[i:i + batch_size]
        remaining = batch

        for attempt in range(max_retries):
            if not remaining:
                break

            batch_input = [{"input_json": json.dumps(remaining, ensure_ascii=False)}]
            raw_response = chain.batch(batch_input)
            returned_items = raw_response[0]

            next_remaining = []
            for original_item, returned_item in zip(remaining, returned_items):
                if is_valid_item(returned_item, parser):
                    results.append(returned_item)
                else:
                    next_remaining.append(original_item)

            valid_count = len(remaining) - len(next_remaining)
            print(f"Batch {i//batch_size+1} | Attempt {attempt+1}: {valid_count} valid, {len(next_remaining)} retry")
            remaining = next_remaining

        if remaining:
            print(f"Batch {i//batch_size+1}: {len(remaining)} failed")

In [11]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", USER_PROMPT_TEMPLATE)
])
chain = prompt_template.partial(format_instructions=format_instructions) | llm | parser 

run_pipeline(chain,  products)

Batch 1 | Attempt 1: 20 valid, 0 retry
Batch 2 | Attempt 1: 20 valid, 0 retry
Batch 3 | Attempt 1: 20 valid, 0 retry
Batch 4 | Attempt 1: 20 valid, 0 retry
Batch 5 | Attempt 1: 20 valid, 0 retry
Batch 6 | Attempt 1: 20 valid, 0 retry
Batch 7 | Attempt 1: 20 valid, 0 retry
Batch 8 | Attempt 1: 20 valid, 0 retry
Batch 9 | Attempt 1: 20 valid, 0 retry
Batch 10 | Attempt 1: 20 valid, 0 retry
Batch 11 | Attempt 1: 20 valid, 0 retry
Batch 12 | Attempt 1: 20 valid, 0 retry
Batch 13 | Attempt 1: 20 valid, 0 retry
Batch 14 | Attempt 1: 20 valid, 0 retry
Batch 15 | Attempt 1: 6 valid, 0 retry


In [13]:
with open("ai_refined_products.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)